# Data centres: base_nodc vs base_dc vs sen_sc_base_dc
Headline comparison of CO2 emissions, total system cost and the average marginal electricity price between the scenario without data centres (`base_nodc`, the reference) and two scenarios with data centres:
- `base_dc`: data centres, flexible P2X loads (electrolysers, electric boilers, heat pumps) not restricted.
- `sen_sc_base_dc`: the same data centres, but with the P2X capacity capped (`TECHGROUPKPOT.inc`), i.e. less flexibility from power-to-X.

Results are the fullweek runs (`MainResults_fullweek.gdx` for `base_nodc` and `base_dc`; `MainResults.gdx` for `sen_sc_base_dc`).
- **CO2 emissions**: `EMI_YCRAG`, all generators and countries (kton).
- **System cost**: `OBJ_YCR`, all cost terms and regions (Mmoney).
- **Marginal electricity price**: `EL_PRICE_YCR` (AVERAGE), regions weighted by their annual electricity demand (`EL_DEMAND_YCR`), as in 13.5.1 of the post-processing notebook (Money/MWh).

In [ ]:
# 0. Settings and helpers
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from pybalmorel import MainResults

ROOT = Path.cwd()                                  # folder containing this notebook (the repository root)
# label -> (scenario folder, results file). The first entry is the reference all changes are measured against.
SCENARIOS = {
    "base_nodc":      ("base_nodc",      "MainResults_fullweek.gdx"),
    "base_dc":        ("base_dc",        "MainResults_fullweek.gdx"),
    "sen_sc_base_dc": ("sen_sc_base_dc", "MainResults.gdx"),       # P2X capacity capped; results file is not named *_fullweek
}
REFERENCE = "base_nodc"
SCEN_COLORS = {"base_nodc": "#9a9992", "base_dc": "#2a78d6", "sen_sc_base_dc": "#eb6834"}

results = {}
for label, (folder, file) in SCENARIOS.items():
    path = ROOT / folder / "model"
    if not (path / file).exists():
        raise FileNotFoundError(f"{file} not found in {path}")
    results[label] = MainResults(files=file, paths=str(path), scenario_names=[label])

# pybalmorel returns some symbols with the raw GAMS column names
RENAME = {"Y": "Year", "C": "Country", "RRR": "Region", "AAA": "Area", "G": "Generation", "FFF": "Fuel",
          "TECH_TYPE": "Technology", "COMMODITY": "Commodity", "CATEGORY": "Category",
          "SUBCATEGORY": "Subcategory", "UNITS": "Unit"}

def scenario_result(label, symbol):
    table = results[label].get_result(symbol).rename(columns=RENAME)
    return table.loc[table["Scenario"].eq(label)].copy()

def solved_years(label, symbol="OBJ_YCR"):
    return set(scenario_result(label, symbol)["Year"].astype(str))

COMPARISON_YEARS = sorted(set.intersection(*(solved_years(l) for l in SCENARIOS)), key=int)
if not COMPARISON_YEARS:
    raise ValueError("No common solved years found for the scenarios")
print("Comparison years:", ", ".join(COMPARISON_YEARS))

def yearly_total(label, symbol):
    """Annual total of a result symbol over all regions/categories, per comparison year."""
    t = scenario_result(label, symbol)
    return t.assign(Year=t["Year"].astype(str)).groupby("Year")["Value"].sum().reindex(COMPARISON_YEARS, fill_value=0)

def price_by(label, by=None):
    """Demand-weighted average marginal electricity price (Money/MWh) per year (and `by`; None = whole system).
    EL_PRICE_YCR (AVERAGE) is the time-averaged marginal cost per region, weighted by annual electricity demand."""
    price = scenario_result(label, "EL_PRICE_YCR")
    price = price.loc[price["Category"].eq("AVERAGE")].assign(Year=lambda d: d["Year"].astype(str))
    dem = scenario_result(label, "EL_DEMAND_YCR").assign(Year=lambda d: d["Year"].astype(str))
    dem = dem.groupby(["Year", "Region"])["Value"].sum().rename("Demand")
    t = price.set_index(["Year", "Region"]).join(dem, how="left").fillna({"Demand": 0}).reset_index()
    keys = ["Year"] + ([by] if by else [])
    t = t.assign(PxD=t["Value"] * t["Demand"]).groupby(keys)[["PxD", "Demand"]].sum()
    out = t["PxD"] / t["Demand"].replace(0, np.nan)
    return out.reindex(COMPARISON_YEARS) if by is None else out

def style(ax, title, ylabel, axis="y"):
    ax.set_title(title, loc="left"); ax.set_ylabel(ylabel); ax.set_xlabel("")
    ax.grid(axis=axis, alpha=0.3); ax.set_axisbelow(True)
    for side in ("top", "right"): ax.spines[side].set_visible(False)

In [ ]:
# 1. CO2 emissions, total system cost and average marginal electricity price: base_nodc vs base_dc vs sen_sc_base_dc
HEADLINE = {
    "CO2 emissions (kton)":                   lambda l: yearly_total(l, "EMI_YCRAG"),
    "Total system cost (Mmoney)":             lambda l: yearly_total(l, "OBJ_YCR"),
    "Marginal electricity price (Money/MWh)": lambda l: price_by(l, None),      # demand-weighted over all regions
}
LABELS = list(SCENARIOS)
SCENS = [l for l in LABELS if l != REFERENCE]                                   # the scenarios compared with the reference
headline = {name: pd.DataFrame({l: f(l) for l in LABELS}) for name, f in HEADLINE.items()}

# Table: value per scenario, plus change and % change of each scenario against the reference
rows = []
for name, t in headline.items():
    for year, r in t.iterrows():
        row = {"Indicator": name, "Year": year, **{l: r[l] for l in LABELS}}
        for l in SCENS:
            row[f"{l}: change"] = r[l] - r[REFERENCE]
            row[f"{l}: change (%)"] = (r[l] - r[REFERENCE]) / r[REFERENCE] * 100 if r[REFERENCE] else np.nan
        rows.append(row)
headline_table = pd.DataFrame(rows).set_index(["Indicator", "Year"])
display(headline_table.round(2))

# Panel per indicator: one bar per scenario for each year, annotated with the % change against the reference
fig, axes = plt.subplots(1, 3, figsize=(max(17, 2.6 * len(COMPARISON_YEARS) + 12), 5.2))
for ax, (name, t) in zip(axes, headline.items()):
    t[LABELS].plot.bar(ax=ax, color=[SCEN_COLORS[l] for l in LABELS], width=0.8, edgecolor="#fcfcfb", legend=False)
    for k, l in enumerate(LABELS):
        if l == REFERENCE: continue
        bars = ax.containers[k]
        for bar, year in zip(bars, t.index):
            pct = headline_table.loc[(name, year), f"{l}: change (%)"]
            ax.annotate(f"{pct:+.1f} %", (bar.get_x() + bar.get_width() / 2, bar.get_height()), ha="center", va="bottom", fontsize=7.5,
                        xytext=(0, 3), textcoords="offset points")
    ax.tick_params(axis="x", rotation=0); ax.margins(y=0.15)
    style(ax, name, name.split("(")[-1].rstrip(")"))
axes[0].legend(frameon=False, loc="upper left")
fig.suptitle(f"Fullweek results; labels = change against {REFERENCE}", x=0.01, ha="left", fontsize=13)
plt.tight_layout(); plt.show()

# Relative change (% of the reference) of the three indicators, per year and scenario
fig, axes = plt.subplots(1, len(COMPARISON_YEARS), figsize=(4.2 * len(COMPARISON_YEARS), 4.2), sharey=True, squeeze=False)
for ax, year in zip(axes[0], COMPARISON_YEARS):
    rel = pd.DataFrame({l: [headline_table.loc[(n, year), f"{l}: change (%)"] for n in HEADLINE] for l in SCENS},
                       index=[n.split(" (")[0] for n in HEADLINE])
    rel.plot.bar(ax=ax, color=[SCEN_COLORS[l] for l in SCENS], width=0.8, edgecolor="#fcfcfb", legend=False)
    for cont in ax.containers:
        ax.bar_label(cont, fmt="%+.1f", fontsize=8, padding=2)
    ax.axhline(0, color="#52514e", linewidth=0.8); ax.tick_params(axis="x", rotation=20); ax.margins(y=0.15)
    style(ax, str(year), "% change vs " + REFERENCE if year == COMPARISON_YEARS[0] else "")
axes[0][0].legend(frameon=False, loc="upper left")
plt.tight_layout(); plt.show()

## 2. Power-to-heat and power-to-hydrogen
How much the two main power-to-X flexibilities change between `base_nodc`, `base_dc` and `sen_sc_base_dc` (the capped-P2X sensitivity). Definitions as in 2.8.1 of the post-processing notebook:
- **Power-to-heat**: heat pumps and electric boilers (technology `ELECT-TO-HEAT`); electricity use = category `ENDOGENOUS_ELECT2HEAT` of `EL_DEMAND_YCR`, capacity from `G_CAP_YCRAF`.
- **Power-to-hydrogen**: electrolysers (technology `ELECTROLYZER`); electricity use = category `ENDO_H2` of `EL_DEMAND_YCR`.
- Top row: electricity used (TWh); bottom row: installed capacity (GW, all capacity categories summed). Labels give the % change against `base_nodc` (for a zero reference, the absolute change).

In [ ]:
# 2. Power-to-heat and power-to-hydrogen: electricity use (TWh) and installed capacity (GW) per scenario and year
P2X = {"Power-to-heat": ("ENDOGENOUS_ELECT2HEAT", "ELECT-TO-HEAT"), "Power-to-hydrogen": ("ENDO_H2", "ELECTROLYZER")}

def p2x_total(label, symbol, column, value):
    t = scenario_result(label, symbol)
    t = t.loc[t[column].eq(value)]
    return t.assign(Year=t["Year"].astype(str)).groupby("Year")["Value"].sum().reindex(COMPARISON_YEARS, fill_value=0)

p2x = {}
for name, (category, technology) in P2X.items():
    p2x[f"{name}: electricity use (TWh)"] = pd.DataFrame({l: p2x_total(l, "EL_DEMAND_YCR", "Category", category) for l in LABELS})
    p2x[f"{name}: capacity (GW)"] = pd.DataFrame({l: p2x_total(l, "G_CAP_YCRAF", "Technology", technology) for l in LABELS})

# Table: value per scenario and change against the reference
rows = []
for name, t in p2x.items():
    for year, r in t.iterrows():
        row = {"Indicator": name, "Year": year, **{l: r[l] for l in LABELS}}
        for l in SCENS:
            row[f"{l}: change"] = r[l] - r[REFERENCE]
            row[f"{l}: change (%)"] = (r[l] - r[REFERENCE]) / r[REFERENCE] * 100 if r[REFERENCE] else np.nan
        rows.append(row)
p2x_table = pd.DataFrame(rows).set_index(["Indicator", "Year"])
display(p2x_table.round(2))

fig, axes = plt.subplots(2, 2, figsize=(max(14, 2.2 * len(COMPARISON_YEARS) + 9), 9))
panel_names = [f"{x}: {m}" for m in ("electricity use (TWh)", "capacity (GW)") for x in P2X]     # row 1: electricity use, row 2: capacity
for ax, name in zip(axes.ravel(), panel_names):
    t = p2x[name]
    t[LABELS].plot.bar(ax=ax, color=[SCEN_COLORS[l] for l in LABELS], width=0.8, edgecolor="#fcfcfb", legend=False)
    for k, l in enumerate(LABELS):
        if l == REFERENCE: continue
        for bar, year in zip(ax.containers[k], t.index):
            ref = t.loc[year, REFERENCE]
            d = t.loc[year, l] - ref
            ax.annotate(f"{d / ref * 100:+.0f} %" if ref else f"{d:+.1f}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                        ha="center", va="bottom", fontsize=7.5, xytext=(0, 3), textcoords="offset points")
    ax.tick_params(axis="x", rotation=0); ax.margins(y=0.15)
    style(ax, name, name.split("(")[-1].rstrip(")"))
axes[0][0].legend(frameon=False, loc="upper left")
fig.suptitle(f"Power-to-X by scenario; labels = change against {REFERENCE}", x=0.01, ha="left", fontsize=13)
plt.tight_layout(); plt.show()

### 2.1 Allowed capacity of heat pumps and electrolysers
The upper limit on installed capacity per region comes from `TECHGROUPKPOT` (MW; existing capacity minus decommissioning plus accumulated new investments of the technology group, in every solved year; `eps` = zero MW, no entry = no limit).
- **`sen_sc_base_dc`**: its `data/TECHGROUPKPOT.inc` limits the groups `HEATPUMP` and `ELECTROLYZER` in every region (the 50 % sensitivity). Electric boilers are not capped.
- **`base_nodc`, `base_dc`**: no such scenario file, so the model default `base/data/TECHGROUPKPOT.inc` applies, which only limits `RESERVOIR_PMP`: heat pumps and electrolysers are **unrestricted**.

First chart: total allowed capacity next to the capacity installed in `CAP_YEAR` (heat pumps = generators `GNR_HP_*`, electrolysers = `GNR_ELYS*`), split into **vintage** capacity (light: `EXOGENOUS` capacity net of `DECOMMISSIONING`, i.e. existing plants still running) and **newly built** capacity (solid: `ENDOGENOUS` investments accumulated up to `CAP_YEAR`). Second chart: the same per country, for the countries with the largest cap.

In [ ]:
# 2.1 Allowed capacity (TECHGROUPKPOT) of heat pumps and electrolysers per scenario, against the installed capacity
import re
CAP_YEAR = COMPARISON_YEARS[-1]                                    # year of the installed capacity
CAP_GROUPS = {"Power-to-heat (heat pumps)": "HEATPUMP", "Power-to-hydrogen (electrolysers)": "ELECTROLYZER"}
CAP_PREFIX = {"HEATPUMP": "GNR_HP_", "ELECTROLYZER": "GNR_ELYS"}   # generator-name prefix of each group, as in NOTEBOOK_sens_sc

def read_techgroupkpot(path):
    """TECHGROUPKPOT table of a GAMS .inc file as region x technology group (MW); eps = 0 MW, missing cell = NaN (no limit).
    Handles the continuation blocks ('+ GROUP1 GROUP2') and skips comments and $ontext/$offtext sections."""
    out, header, skip = {}, None, False
    for line in Path(path).read_text(encoding="utf-8", errors="ignore").splitlines():
        s = line.strip()
        if s.lower().startswith("$ontext"): skip = True; continue
        if s.lower().startswith("$offtext"): skip = False; continue
        if skip or not s or s.startswith("*") or s.startswith(";") or s.upper().startswith("TABLE"): continue
        if s.startswith("+") or header is None:
            body = line.replace("+", " ", 1) if s.startswith("+") else line
            header = [(m.group(0), m.start()) for m in re.finditer(r"\S+", body)]
            continue
        tokens = [(m.group(0), m.start()) for m in re.finditer(r"\S+", line)]
        region, cells = tokens[0][0], tokens[1:]
        for text, pos in cells:
            group = min(header, key=lambda h: abs(h[1] - pos))[0]
            out.setdefault(region, {})[group] = 0.0 if text.lower() == "eps" else float(text)
    return pd.DataFrame(out).T

def scenario_kpot(folder):
    """Cap table in force for a scenario: its own data/TECHGROUPKPOT.inc, otherwise the model default base/data one."""
    own = ROOT / folder / "data" / "TECHGROUPKPOT.inc"
    return read_techgroupkpot(own if own.exists() else ROOT / "base" / "data" / "TECHGROUPKPOT.inc"), own.exists()

kpot = {l: scenario_kpot(folder) for l, (folder, _) in SCENARIOS.items()}
allowed = {}                                                       # label -> group -> region Series (GW); NaN/absent = unrestricted
for l, (table, own) in kpot.items():
    allowed[l] = {g: (table[g].dropna() / 1e3 if g in table else pd.Series(dtype=float)) for g in CAP_GROUPS.values()}
    print(f"{l}: {'own' if own else 'default base'} TECHGROUPKPOT, groups limited: {sorted(table.columns)}")

# Installed capacity (GW) per scenario, region and group, split into vintage and newly built capacity:
#   vintage = EXOGENOUS capacity net of DECOMMISSIONING (existing plants still running), new = ENDOGENOUS (model investments, accumulated up to CAP_YEAR)
# The column holding the capacity category is named differently across pybalmorel versions: found by its values (as in NOTEBOOK_sens_sc).
def capacity_category_column(t):
    return next(col for col in t.columns if t[col].isin(["ENDOGENOUS"]).any())

def installed(label, group):
    t = scenario_result(label, "G_CAP_YCRAF")
    t = t.loc[t["Year"].astype(str).eq(CAP_YEAR) & t["Generation"].astype(str).str.upper().str.startswith(CAP_PREFIX[group])]
    t = t.assign(Kind=np.where(t[capacity_category_column(t)].eq("ENDOGENOUS"), "New", "Vintage"))   # EXOGENOUS + DECOMMISSIONING -> vintage
    return t.groupby(["Country", "Region", "Kind"])["Value"].sum().unstack("Kind", fill_value=0).reindex(columns=["Vintage", "New"], fill_value=0)
inst = {(l, g): installed(l, g) for l in LABELS for g in CAP_GROUPS.values()}
region_country = pd.concat(inst.values()).reset_index().drop_duplicates("Region").set_index("Region")["Country"]

# Table: allowed (sum over limited regions) and installed capacity in GW
rows = []
for title, g in CAP_GROUPS.items():
    for l in LABELS:
        a = allowed[l][g]
        rows.append({"Technology": title, "Scenario": l, "Allowed (GW)": a.sum() if len(a) else np.nan, "Regions with a limit": len(a),
                     f"Vintage {CAP_YEAR} (GW)": inst[(l, g)]["Vintage"].sum(),
                     f"New {CAP_YEAR} (GW)": inst[(l, g)]["New"].sum(), f"Installed {CAP_YEAR} (GW)": inst[(l, g)].to_numpy().sum()})
cap_table = pd.DataFrame(rows).set_index(["Technology", "Scenario"])
display(cap_table.round(1))

# Chart 1: total allowed capacity (hatched) vs installed capacity, stacked into vintage (light) and newly built (solid)
from matplotlib.patches import Patch
def light(color, f=0.55):
    r, g_, b_ = (int(color[i:i + 2], 16) for i in (1, 3, 5))
    return "#%02x%02x%02x" % tuple(int(v + (255 - v) * f) for v in (r, g_, b_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
x = np.arange(len(LABELS)); w = 0.38
for ax, (title, g) in zip(axes, CAP_GROUPS.items()):
    a = [allowed[l][g].sum() if len(allowed[l][g]) else 0 for l in LABELS]
    vint = [inst[(l, g)]["Vintage"].sum() for l in LABELS]
    new = [inst[(l, g)]["New"].sum() for l in LABELS]
    ax.bar(x - w / 2, a, w, color=[SCEN_COLORS[l] for l in LABELS], edgecolor="#fcfcfb", hatch="//")
    ax.bar(x + w / 2, vint, w, color=[light(SCEN_COLORS[l]) for l in LABELS], edgecolor="#fcfcfb")
    ax.bar(x + w / 2, new, w, bottom=vint, color=[SCEN_COLORS[l] for l in LABELS], edgecolor="#fcfcfb")
    for xi, l, v in zip(x, LABELS, a):
        ax.annotate(f"{v:,.0f}" if len(allowed[l][g]) else "no limit", (xi - w / 2, v), ha="center", va="bottom", fontsize=8,
                    xytext=(0, 3), textcoords="offset points")
    for xi, v, n in zip(x, vint, new):
        if v > 0: ax.text(xi + w / 2, v / 2, f"{v:,.0f}", ha="center", va="center", fontsize=7.5)
        if n > 0: ax.text(xi + w / 2, v + n / 2, f"{n:,.0f}", ha="center", va="center", fontsize=7.5, color="white")
        ax.annotate(f"{v + n:,.0f}", (xi + w / 2, v + n), ha="center", va="bottom", fontsize=8, xytext=(0, 3), textcoords="offset points")
    ax.set_xticks(x); ax.set_xticklabels(LABELS); ax.margins(y=0.15)
    style(ax, title, "GW")
axes[0].legend(handles=[Patch(facecolor="#9a9992", edgecolor="#52514e", hatch="//", label="Allowed (TECHGROUPKPOT)"),
                        Patch(facecolor=light("#52514e"), edgecolor="#52514e", label=f"Installed {CAP_YEAR}: vintage"),
                        Patch(facecolor="#52514e", edgecolor="#52514e", label=f"Installed {CAP_YEAR}: newly built")], frameon=False, loc="upper left", fontsize=8)
fig.suptitle("Allowed capacity (upper limit per region, summed) and installed capacity, vintage vs newly built", x=0.01, ha="left", fontsize=13)
plt.tight_layout(); plt.show()

# Chart 2: per country, the 12 countries with the largest allowed capacity: the limit (grey) and the installed capacity of each scenario
TOP = 12
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (title, g) in zip(axes, CAP_GROUPS.items()):
    lim = allowed["sen_sc_base_dc"][g].groupby(region_country).sum()
    top = lim.sort_values(ascending=False).head(TOP).index[::-1]
    y = np.arange(len(top)); h = 0.2
    ax.barh(y + 1.5 * h, lim.reindex(top), h, color="#d6d5d0", edgecolor="#52514e", hatch="//")
    for k, l in enumerate(LABELS):
        t = inst[(l, g)].groupby("Country").sum().reindex(top).fillna(0)
        ax.barh(y + (0.5 - k) * h, t["Vintage"], h, color=light(SCEN_COLORS[l]), edgecolor="#fcfcfb")
        ax.barh(y + (0.5 - k) * h, t["New"], h, left=t["Vintage"], color=SCEN_COLORS[l], edgecolor="#fcfcfb")
    ax.set_yticks(y); ax.set_yticklabels([str(c).replace("_", " ").title() for c in top])
    style(ax, title, "", axis="x"); ax.set_xlabel("GW")
axes[0].legend(handles=[Patch(facecolor="#d6d5d0", edgecolor="#52514e", hatch="//", label="Allowed (sen_sc)")]
                       + [Patch(facecolor=SCEN_COLORS[l], label=f"{l}: newly built (light = vintage)") for l in LABELS], frameon=False, loc="lower right", fontsize=8)
fig.suptitle(f"Allowed capacity in sen_sc_base_dc and installed capacity (vintage + new) in {CAP_YEAR}, top {TOP} countries by cap (base_nodc and base_dc are unrestricted)", x=0.01, ha="left", fontsize=12)
plt.tight_layout(); plt.show()